# 第131章: Video Diffusion Models — 画像拡散モデルを動画に拡張する

## 📋 この章で学ぶこと

この章を終えると、以下ができるようになります：

- [ ] U-Netにtemporal層を挿入して動画拡散モデルを構築できる
- [ ] (2+1)D Convolution の仕組みと利点を説明できる
- [ ] 動画用ノイズスケジュールを設計できる
- [ ] Moving-MNISTデータセットを合成し、動画拡散モデルを訓練できる
- [ ] 生成した動画をGIFで可視化できる

## 🎯 前提知識

- ✅ Notebook 130（時間的注意機構の基礎）
- ✅ Notebook 40-42（DDPM / CFG）
- ✅ Notebook 39（U-Net / 位置エンコーディング）

⏱️ **推定学習時間**: 150-180分  
📊 **難易度**: ★★★★☆（上級）  
🎓 **カテゴリ**: 時空間モデリング

## 目次

1. [画像拡散から動画拡散へ](#section1)
2. [(2+1)D Convolutionの仕組み](#section2)
3. [TemporalUNetBlockの実装](#section3)
4. [VideoUNetの構築](#section4)
5. [Moving-MNISTデータセット](#section5)
6. [動画拡散モデルの訓練](#section6)
7. [動画生成とGIF可視化](#section7)
8. [まとめと自己評価クイズ](#summary)

In [ ]:
# ============================================================
# 環境設定
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import warnings
import math

warnings.filterwarnings('ignore')

def setup_japanese_font():
    japanese_fonts = ['Hiragino Sans', 'Yu Gothic', 'Noto Sans CJK JP', 'IPAexGothic']
    available = set(f.name for f in fm.fontManager.ttflist)
    for f in japanese_fonts:
        if f in available:
            plt.rcParams['font.family'] = f
            plt.rcParams['axes.unicode_minus'] = False
            return f
    return None

font_used = setup_japanese_font()
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else
                       'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'✅ ライブラリのインポート完了')
print(f'🖥️ デバイス: {device}')

<a id="section1"></a>
## 1. 画像拡散から動画拡散へ

### 🤔 DDPMの復習と動画への拡張

Notebook 40-42で学んだDDPMは、1枚の画像 $x_0$ にガウスノイズを徐々に加え、
完全なノイズ $x_T$ から逆拡散で画像を復元するモデルでした。

**動画拡散モデル**は同じ原理を拡張します：

| 画像拡散 | 動画拡散 |
|---------|--------|
| $x_0 \in \mathbb{R}^{C \times H \times W}$ | $x_0 \in \mathbb{R}^{T \times C \times H \times W}$ |
| U-Net: 2D Conv | U-Net: 2D Conv + Temporal Attention |
| 空間的整合性のみ | 空間的 + 時間的整合性 |
| 1枚の画像を生成 | T枚のフレーム列を一括生成 |

鍵となるアイデアは、**既存の画像U-Netにtemporal層を追加する**ことです。
これにより、画像生成の能力を保ちつつ、時間的な一貫性を学習できます。

In [ ]:
# ============================================================
# 画像拡散と動画拡散の比較図
# ============================================================

def visualize_image_vs_video_diffusion():
    """画像拡散モデルと動画拡散モデルの比較"""
    fig, axes = plt.subplots(2, 5, figsize=(18, 7))
    
    timesteps = [0, 25, 50, 75, 100]
    
    # 画像拡散（上段）
    np.random.seed(42)
    img = np.zeros((32, 32))
    img[8:24, 8:24] = 1.0  # 白い四角
    
    for i, t in enumerate(timesteps):
        noise_level = t / 100.0
        noisy = img * (1 - noise_level) + np.random.randn(32, 32) * noise_level
        axes[0, i].imshow(noisy, cmap='gray', vmin=-1, vmax=2)
        axes[0, i].set_title(f't={t}', fontsize=11)
        axes[0, i].axis('off')
    axes[0, 0].set_ylabel('画像拡散\n(1枚)', fontsize=12, fontweight='bold')
    
    # 動画拡散（下段） - 4フレームの動画
    for i, t in enumerate(timesteps):
        noise_level = t / 100.0
        # 4フレームを横に並べて表示
        frames = np.zeros((32, 32 * 4 + 3))
        for f in range(4):
            offset = int(4 * f)  # 四角形が移動する
            frame = np.zeros((32, 32))
            frame[8:24, (8+offset):(24+offset)] = 1.0
            noisy_frame = frame * (1 - noise_level) + np.random.randn(32, 32) * noise_level
            frames[:, f*35:f*35+32] = noisy_frame
        axes[1, i].imshow(frames, cmap='gray', vmin=-1, vmax=2)
        axes[1, i].set_title(f't={t}', fontsize=11)
        axes[1, i].axis('off')
    axes[1, 0].set_ylabel('動画拡散\n(4フレーム)', fontsize=12, fontweight='bold')
    
    plt.suptitle('拡散過程: 画像 vs 動画\n← クリーン | ノイズ →', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualize_image_vs_video_diffusion()

<a id="section2"></a>
## 2. (2+1)D Convolution の仕組み

### 📊 3D Conv vs (2+1)D Conv

動画を処理する畳み込みには主に2つのアプローチがあります：

1. **3D Convolution**: 空間(H,W)と時間(T)を同時に畳み込む
2. **(2+1)D Convolution**: まず空間2Dで畳み込み、次に時間1Dで畳み込む（分離）

(2+1)D の利点：
- パラメータ数が少ない
- 学習が安定
- 画像の事前学習済み重みを活用できる

In [ ]:
# ============================================================
# (2+1)D Convolution の実装と比較
# ============================================================

class Conv2Plus1D(nn.Module):
    """(2+1)D Convolution: 空間2D + 時間1D の分離畳み込み"""
    
    def __init__(self, in_channels, out_channels, kernel_size=3, padding=1):
        super().__init__()
        # 中間チャンネル数（パラメータ数を3D Convと同程度に）
        mid_channels = (in_channels * out_channels * 3 * kernel_size * kernel_size) // \
                       (in_channels * kernel_size * kernel_size + 3 * out_channels)
        mid_channels = max(mid_channels, 1)
        
        # 空間方向の2D畳み込み
        self.spatial_conv = nn.Conv2d(
            in_channels, mid_channels, 
            kernel_size=(kernel_size, kernel_size), 
            padding=(padding, padding)
        )
        self.bn1 = nn.BatchNorm2d(mid_channels)
        
        # 時間方向の1D畳み込み
        self.temporal_conv = nn.Conv1d(
            mid_channels, out_channels,
            kernel_size=3, padding=1
        )
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x):
        """x: (B, C, T, H, W) → (B, C_out, T, H, W)"""
        B, C, T, H, W = x.shape
        
        # 空間畳み込み: 各フレームに独立に適用
        x = x.permute(0, 2, 1, 3, 4).reshape(B * T, C, H, W)
        x = self.relu(self.bn1(self.spatial_conv(x)))
        C_mid = x.shape[1]
        x = x.reshape(B, T, C_mid, H, W).permute(0, 2, 1, 3, 4)
        
        # 時間畳み込み: 各空間位置に独立に適用
        x = x.reshape(B, C_mid, T, H * W)
        x = x.permute(0, 3, 1, 2).reshape(B * H * W, C_mid, T)
        x = self.relu(self.bn2(self.temporal_conv(x)))
        C_out = x.shape[1]
        x = x.reshape(B, H * W, C_out, T).permute(0, 2, 3, 1)
        x = x.reshape(B, C_out, T, H, W)
        
        return x

# 3D Conv vs (2+1)D Conv のパラメータ比較
in_ch, out_ch = 32, 64

conv3d = nn.Conv3d(in_ch, out_ch, kernel_size=3, padding=1)
conv2plus1d = Conv2Plus1D(in_ch, out_ch)

params_3d = sum(p.numel() for p in conv3d.parameters())
params_2plus1d = sum(p.numel() for p in conv2plus1d.parameters())

print("="*60)
print("3D Conv vs (2+1)D Conv")
print("="*60)
print(f"3D Conv パラメータ: {params_3d:,}")
print(f"(2+1)D Conv パラメータ: {params_2plus1d:,}")
print(f"比率: {params_2plus1d / params_3d:.2f}")

# 動作確認
x = torch.randn(2, in_ch, 8, 16, 16)  # (B, C, T, H, W)
print(f"\n入力: {x.shape}")
print(f"3D Conv 出力: {conv3d(x).shape}")
print(f"(2+1)D Conv 出力: {conv2plus1d(x).shape}")

<a id="section3"></a>
## 3. TemporalUNetBlockの実装

### 📊 U-Netにtemporal層を追加する

画像用U-Netの各ブロックに、時間方向の処理を追加します：

```
[2D Conv + GroupNorm + SiLU]   ← 空間的特徴抽出（既存）
         ↓
[Temporal Conv1D]              ← 時間的な混合（新規追加）
         ↓
[Temporal Self-Attention]      ← 時間的な注意機構（新規追加）
```

In [ ]:
# ============================================================
# 時間的位置エンコーディング（Sinusoidal）
# ============================================================

def sinusoidal_embedding(timesteps, dim):
    """拡散ステップの正弦波位置エンコーディング
    
    Args:
        timesteps: (B,) の整数テンソル
        dim: 埋め込み次元
    Returns:
        (B, dim) の埋め込みテンソル
    """
    half_dim = dim // 2
    emb = math.log(10000) / (half_dim - 1)
    emb = torch.exp(torch.arange(half_dim, device=timesteps.device) * -emb)
    emb = timesteps[:, None].float() * emb[None, :]
    emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=-1)
    return emb

# テスト
t = torch.tensor([0, 50, 100, 200, 500])
emb = sinusoidal_embedding(t, 64)
print(f"タイムステップ埋め込み: {t.shape} → {emb.shape}")

In [ ]:
# ============================================================
# TemporalUNetBlock の実装
# ============================================================

class TemporalUNetBlock(nn.Module):
    """時間的注意機構付きU-Netブロック
    
    2D空間畳み込み + 時間畳み込み + 時間的Attention
    """
    
    def __init__(self, in_ch, out_ch, time_emb_dim, num_frames, num_heads=4):
        super().__init__()
        self.num_frames = num_frames
        
        # 空間2D畳み込み
        self.spatial_block = nn.Sequential(
            nn.GroupNorm(8, in_ch),
            nn.SiLU(),
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.GroupNorm(8, out_ch),
            nn.SiLU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
        )
        
        # タイムステップ埋め込みの射影
        self.time_mlp = nn.Sequential(
            nn.SiLU(),
            nn.Linear(time_emb_dim, out_ch),
        )
        
        # 時間方向1D畳み込み
        self.temporal_conv = nn.Sequential(
            nn.GroupNorm(8, out_ch),
            nn.SiLU(),
            nn.Conv1d(out_ch, out_ch, kernel_size=3, padding=1),
        )
        
        # 時間的Self-Attention
        self.temporal_attn = nn.MultiheadAttention(
            embed_dim=out_ch, num_heads=num_heads, batch_first=True
        )
        self.temporal_norm = nn.LayerNorm(out_ch)
        
        # 残差接続用
        self.residual_conv = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    
    def forward(self, x, t_emb):
        """順伝播
        
        Args:
            x: (B*T, C, H, W) 入力特徴
            t_emb: (B, time_emb_dim) タイムステップ埋め込み
        Returns:
            (B*T, C_out, H, W) 出力特徴
        """
        BT, C, H, W = x.shape
        B = BT // self.num_frames
        T = self.num_frames
        
        # 残差
        residual = self.residual_conv(x)
        
        # 1. 空間2D畳み込み
        h = self.spatial_block(x)  # (B*T, C_out, H, W)
        
        # 2. タイムステップ条件付け
        t_proj = self.time_mlp(t_emb)  # (B, C_out)
        t_proj = t_proj.unsqueeze(1).repeat(1, T, 1)  # (B, T, C_out)
        t_proj = t_proj.reshape(B * T, -1, 1, 1)  # (B*T, C_out, 1, 1)
        h = h + t_proj
        
        C_out = h.shape[1]
        
        # 3. 時間方向1D畳み込み
        # (B*T, C, H, W) → (B, C, T, H*W) → temporal conv → 元に戻す
        h_temp = h.reshape(B, T, C_out, H, W).permute(0, 2, 1, 3, 4)  # (B, C, T, H, W)
        h_temp = h_temp.reshape(B * C_out, T, H * W)  # 時間方向にconv1d
        # conv1dは (N, C, L) を期待するのでH*Wをチャンネルに
        h_temp = h.reshape(B, T, C_out, H * W).permute(0, 3, 2, 1)  # (B, H*W, C, T)
        h_temp = h_temp.reshape(B * H * W, C_out, T)
        h_temp = self.temporal_conv(h_temp)  # (B*H*W, C, T)
        h_temp = h_temp.reshape(B, H * W, C_out, T).permute(0, 3, 2, 1)  # (B, T, C, H*W)
        h_temp = h_temp.reshape(B * T, C_out, H, W)
        h = h + h_temp
        
        # 4. 時間的Self-Attention
        # (B*T, C, H, W) → (B*H*W, T, C)
        h_attn = h.reshape(B, T, C_out, H * W).permute(0, 3, 1, 2)  # (B, H*W, T, C)
        h_attn = h_attn.reshape(B * H * W, T, C_out)
        h_attn_norm = self.temporal_norm(h_attn)
        h_attn_out, _ = self.temporal_attn(h_attn_norm, h_attn_norm, h_attn_norm)
        h_attn = h_attn + h_attn_out  # 残差接続
        # (B*H*W, T, C) → (B*T, C, H, W)
        h_attn = h_attn.reshape(B, H * W, T, C_out).permute(0, 2, 3, 1)  # (B, T, C, H*W)
        h_attn = h_attn.reshape(B * T, C_out, H, W)
        h = h + h_attn
        
        return h + residual

# テスト
B, T, C, H, W = 2, 8, 32, 16, 16
x = torch.randn(B * T, C, H, W)
t_emb = torch.randn(B, 64)

block = TemporalUNetBlock(in_ch=32, out_ch=64, time_emb_dim=64, num_frames=T)
out = block(x, t_emb)
print(f"TemporalUNetBlock: {x.shape} → {out.shape} ✅")

<a id="section4"></a>
## 4. VideoUNetの構築

### 📊 全体アーキテクチャ

```
入力: (B, T, C, H, W) + タイムステップ t
  ↓
[Encoder] TemporalUNetBlock × 3 + Downsample
  ↓
[Bottleneck] TemporalUNetBlock
  ↓
[Decoder] TemporalUNetBlock × 3 + Upsample + Skip Connections
  ↓
出力: (B, T, C, H, W) — 予測ノイズ
```

In [ ]:
# ============================================================
# VideoUNet の実装
# ============================================================

class VideoUNet(nn.Module):
    """動画拡散モデル用U-Net
    
    画像U-Netをベースに、各ブロックにtemporal層を追加。
    小規模なMoving-MNIST (32x32) 向けの軽量設計。
    """
    
    def __init__(self, in_channels=1, num_frames=16, base_dim=32, time_emb_dim=128):
        super().__init__()
        self.num_frames = num_frames
        
        # タイムステップ埋め込み
        self.time_mlp = nn.Sequential(
            nn.Linear(time_emb_dim, time_emb_dim),
            nn.SiLU(),
            nn.Linear(time_emb_dim, time_emb_dim),
        )
        self.time_emb_dim = time_emb_dim
        
        # 入力畳み込み
        self.input_conv = nn.Conv2d(in_channels, base_dim, 3, padding=1)
        
        # エンコーダ
        self.enc1 = TemporalUNetBlock(base_dim, base_dim, time_emb_dim, num_frames)
        self.down1 = nn.Conv2d(base_dim, base_dim, 4, stride=2, padding=1)
        
        self.enc2 = TemporalUNetBlock(base_dim, base_dim * 2, time_emb_dim, num_frames)
        self.down2 = nn.Conv2d(base_dim * 2, base_dim * 2, 4, stride=2, padding=1)
        
        # ボトルネック
        self.bottleneck = TemporalUNetBlock(base_dim * 2, base_dim * 2, time_emb_dim, num_frames)
        
        # デコーダ
        self.up2 = nn.ConvTranspose2d(base_dim * 2, base_dim * 2, 4, stride=2, padding=1)
        self.dec2 = TemporalUNetBlock(base_dim * 4, base_dim, time_emb_dim, num_frames)  # skip
        
        self.up1 = nn.ConvTranspose2d(base_dim, base_dim, 4, stride=2, padding=1)
        self.dec1 = TemporalUNetBlock(base_dim * 2, base_dim, time_emb_dim, num_frames)  # skip
        
        # 出力
        self.output_conv = nn.Sequential(
            nn.GroupNorm(8, base_dim),
            nn.SiLU(),
            nn.Conv2d(base_dim, in_channels, 3, padding=1),
        )
    
    def forward(self, x, t):
        """順伝播
        
        Args:
            x: (B, T, C, H, W) ノイズ付き動画
            t: (B,) 拡散タイムステップ
        Returns:
            (B, T, C, H, W) 予測ノイズ
        """
        B, T, C_in, H, W = x.shape
        
        # タイムステップ埋め込み
        t_emb = sinusoidal_embedding(t, self.time_emb_dim)
        t_emb = self.time_mlp(t_emb)  # (B, time_emb_dim)
        
        # (B, T, C, H, W) → (B*T, C, H, W) で空間処理
        x = x.reshape(B * T, C_in, H, W)
        x = self.input_conv(x)
        
        # エンコーダ
        e1 = self.enc1(x, t_emb)         # (B*T, base, H, W)
        x = self.down1(e1)               # (B*T, base, H/2, W/2)
        
        e2 = self.enc2(x, t_emb)         # (B*T, base*2, H/2, W/2)
        x = self.down2(e2)               # (B*T, base*2, H/4, W/4)
        
        # ボトルネック
        x = self.bottleneck(x, t_emb)    # (B*T, base*2, H/4, W/4)
        
        # デコーダ (Skip connections)
        x = self.up2(x)                  # (B*T, base*2, H/2, W/2)
        x = torch.cat([x, e2], dim=1)    # (B*T, base*4, H/2, W/2)
        x = self.dec2(x, t_emb)          # (B*T, base, H/2, W/2)
        
        x = self.up1(x)                  # (B*T, base, H, W)
        x = torch.cat([x, e1], dim=1)    # (B*T, base*2, H, W)
        x = self.dec1(x, t_emb)          # (B*T, base, H, W)
        
        # 出力
        x = self.output_conv(x)          # (B*T, C_in, H, W)
        x = x.reshape(B, T, C_in, H, W)
        
        return x

# テスト
model = VideoUNet(in_channels=1, num_frames=8, base_dim=32)
x = torch.randn(2, 8, 1, 32, 32)
t = torch.randint(0, 1000, (2,))
out = model(x, t)

total_params = sum(p.numel() for p in model.parameters())
print(f"VideoUNet テスト:")
print(f"  入力: {x.shape}")
print(f"  出力: {out.shape}")
print(f"  パラメータ数: {total_params:,}")
print("✅ VideoUNet 動作確認完了")

<a id="section5"></a>
## 5. Moving-MNISTデータセット

### 📊 合成データによる動画生成

Moving-MNISTは、MNIST数字が32×32のキャンバス上を等速直線運動する動画データセットです。
ダウンロード不要で、プログラムから直接生成できます。

各クリップは16フレーム、32×32ピクセルのグレースケール動画です。

In [ ]:
# ============================================================
# Moving-MNIST データセットの合成
# ============================================================

class MovingMNISTDataset(Dataset):
    """Moving-MNIST: MNIST数字が移動する動画データセット
    
    合成データなのでダウンロード不要。
    """
    
    def __init__(self, num_clips=2000, num_frames=16, canvas_size=32, digit_size=14):
        super().__init__()
        self.num_clips = num_clips
        self.num_frames = num_frames
        self.canvas_size = canvas_size
        self.digit_size = digit_size
        
        # 簡易数字テンプレート（手書き風の丸い形状）
        self.digits = self._create_digit_templates()
        
        # クリップを事前生成
        self.clips = self._generate_all_clips()
    
    def _create_digit_templates(self):
        """簡易的な数字テンプレートを生成"""
        templates = []
        for digit in range(10):
            img = np.zeros((self.digit_size, self.digit_size))
            # 各数字に対して異なる形状を生成
            np.random.seed(digit * 100)
            cx, cy = self.digit_size // 2, self.digit_size // 2
            for _ in range(30):
                dx = int(np.random.randn() * 3)
                dy = int(np.random.randn() * 3)
                px = np.clip(cx + dx, 0, self.digit_size - 1)
                py = np.clip(cy + dy, 0, self.digit_size - 1)
                img[px, py] = 1.0
                # 太くする
                for ddx in [-1, 0, 1]:
                    for ddy in [-1, 0, 1]:
                        px2 = np.clip(px + ddx, 0, self.digit_size - 1)
                        py2 = np.clip(py + ddy, 0, self.digit_size - 1)
                        img[px2, py2] = max(img[px2, py2], 0.7)
            templates.append(img)
        return templates
    
    def _generate_clip(self, idx):
        """1クリップの動画を生成"""
        rng = np.random.RandomState(idx)
        frames = np.zeros((self.num_frames, self.canvas_size, self.canvas_size))
        
        # ランダムな数字を選択
        digit_idx = rng.randint(0, 10)
        digit = self.digits[digit_idx]
        
        # 初期位置と速度
        max_pos = self.canvas_size - self.digit_size
        x = rng.uniform(0, max_pos)
        y = rng.uniform(0, max_pos)
        vx = rng.uniform(-2, 2)
        vy = rng.uniform(-2, 2)
        
        for t in range(self.num_frames):
            # 壁での反射
            if x < 0 or x > max_pos:
                vx = -vx
                x = np.clip(x, 0, max_pos)
            if y < 0 or y > max_pos:
                vy = -vy
                y = np.clip(y, 0, max_pos)
            
            # 数字をキャンバスに配置
            xi, yi = int(round(x)), int(round(y))
            frames[t, yi:yi+self.digit_size, xi:xi+self.digit_size] = digit
            
            # 移動
            x += vx
            y += vy
        
        return frames
    
    def _generate_all_clips(self):
        """全クリップを事前生成"""
        clips = []
        for i in range(self.num_clips):
            clip = self._generate_clip(i)
            clips.append(torch.tensor(clip, dtype=torch.float32))
        return clips
    
    def __len__(self):
        return self.num_clips
    
    def __getitem__(self, idx):
        # (T, H, W) → (T, 1, H, W), 値域を[-1, 1]に
        clip = self.clips[idx].unsqueeze(1) * 2.0 - 1.0
        return clip

# データセット生成
dataset = MovingMNISTDataset(num_clips=2000, num_frames=16, canvas_size=32)
print(f"データセット: {len(dataset)} クリップ")
print(f"各クリップの形状: {dataset[0].shape}")

In [ ]:
# ============================================================
# データセットの可視化
# ============================================================

def show_video_clips(dataset, num_clips=4, frames_to_show=8):
    """動画クリップを可視化"""
    fig, axes = plt.subplots(num_clips, frames_to_show, figsize=(16, 2 * num_clips))
    
    for i in range(num_clips):
        clip = dataset[i]  # (T, 1, H, W)
        step = max(1, len(clip) // frames_to_show)
        for j in range(frames_to_show):
            t = j * step
            if t >= len(clip):
                t = len(clip) - 1
            axes[i, j].imshow(clip[t, 0].numpy(), cmap='gray', vmin=-1, vmax=1)
            axes[i, j].set_title(f't={t}', fontsize=9)
            axes[i, j].axis('off')
        axes[i, 0].set_ylabel(f'Clip {i}', fontsize=10)
    
    plt.suptitle('Moving-MNIST サンプル', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

show_video_clips(dataset)

<a id="section6"></a>
## 6. 動画拡散モデルの訓練

### 📊 拡散過程の設定

画像DDPMと同様に、ノイズスケジュールに基づいて動画にノイズを加え、
U-Netでノイズを予測するよう訓練します。

In [ ]:
# ============================================================
# 動画拡散モデル (Video Diffusion)
# ============================================================

class VideoDiffusion:
    """動画拡散モデルの訓練・サンプリング"""
    
    def __init__(self, model, num_timesteps=500, device='cpu'):
        self.model = model
        self.num_timesteps = num_timesteps
        self.device = device
        
        # Cosineノイズスケジュール
        self.betas = self._cosine_beta_schedule(num_timesteps).to(device)
        self.alphas = (1 - self.betas).to(device)
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0).to(device)
        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod).to(device)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1 - self.alphas_cumprod).to(device)
    
    def _cosine_beta_schedule(self, T, s=0.008):
        """Cosineノイズスケジュール"""
        steps = torch.arange(T + 1, dtype=torch.float32) / T
        alphas_bar = torch.cos((steps + s) / (1 + s) * math.pi / 2) ** 2
        alphas_bar = alphas_bar / alphas_bar[0]
        betas = 1 - alphas_bar[1:] / alphas_bar[:-1]
        return torch.clamp(betas, 0.0001, 0.02)
    
    def q_sample(self, x_0, t, noise=None):
        """前方拡散: x_0にノイズを加える
        
        Args:
            x_0: (B, T, C, H, W) クリーン動画
            t: (B,) タイムステップ
            noise: 追加するノイズ
        Returns:
            x_t: ノイズ付き動画
        """
        if noise is None:
            noise = torch.randn_like(x_0)
        
        sqrt_alpha = self.sqrt_alphas_cumprod[t][:, None, None, None, None]
        sqrt_one_minus_alpha = self.sqrt_one_minus_alphas_cumprod[t][:, None, None, None, None]
        
        return sqrt_alpha * x_0 + sqrt_one_minus_alpha * noise
    
    def train_step(self, x_0, optimizer):
        """1ステップの訓練"""
        B = x_0.shape[0]
        t = torch.randint(0, self.num_timesteps, (B,), device=self.device)
        noise = torch.randn_like(x_0)
        
        x_t = self.q_sample(x_0, t, noise)
        predicted_noise = self.model(x_t, t)
        
        loss = F.mse_loss(predicted_noise, noise)
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
        optimizer.step()
        
        return loss.item()
    
    @torch.no_grad()
    def sample(self, shape):
        """逆拡散によるサンプリング
        
        Args:
            shape: (B, T, C, H, W)
        Returns:
            生成動画
        """
        self.model.eval()
        x = torch.randn(shape, device=self.device)
        
        for t in reversed(range(self.num_timesteps)):
            t_batch = torch.full((shape[0],), t, device=self.device, dtype=torch.long)
            
            predicted_noise = self.model(x, t_batch)
            
            alpha = self.alphas[t]
            alpha_bar = self.alphas_cumprod[t]
            beta = self.betas[t]
            
            x = (1 / torch.sqrt(alpha)) * (x - beta / torch.sqrt(1 - alpha_bar) * predicted_noise)
            
            if t > 0:
                noise = torch.randn_like(x)
                x = x + torch.sqrt(beta) * noise
        
        self.model.train()
        return x.clamp(-1, 1)

print("✅ VideoDiffusion クラス定義完了")

In [ ]:
# ============================================================
# 訓練の実行
# ============================================================

# ハイパーパラメータ（教育目的で小規模に）
NUM_FRAMES = 8
BATCH_SIZE = 8
NUM_EPOCHS = 15
LR = 1e-3

# データセット（小規模版）
train_dataset = MovingMNISTDataset(num_clips=500, num_frames=NUM_FRAMES, canvas_size=32)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

# モデル
video_unet = VideoUNet(
    in_channels=1,
    num_frames=NUM_FRAMES,
    base_dim=32,
    time_emb_dim=64
).to(device)

optimizer = torch.optim.Adam(video_unet.parameters(), lr=LR)
diffusion = VideoDiffusion(video_unet, num_timesteps=300, device=device)

total_params = sum(p.numel() for p in video_unet.parameters())
print(f"モデルパラメータ: {total_params:,}")
print(f"デバイス: {device}")
print(f"訓練開始...")

# 訓練ループ
losses = []
for epoch in range(NUM_EPOCHS):
    epoch_losses = []
    for batch in train_loader:
        batch = batch.to(device)  # (B, T, 1, H, W)
        loss = diffusion.train_step(batch, optimizer)
        epoch_losses.append(loss)
    
    avg_loss = np.mean(epoch_losses)
    losses.append(avg_loss)
    if (epoch + 1) % 5 == 0:
        print(f"  Epoch {epoch+1}/{NUM_EPOCHS}, Loss: {avg_loss:.4f}")

print("✅ 訓練完了")

In [ ]:
# ============================================================
# 訓練損失の可視化
# ============================================================

plt.figure(figsize=(10, 5))
plt.plot(losses, 'b-', linewidth=2)
plt.xlabel('エポック', fontsize=12)
plt.ylabel('損失 (MSE)', fontsize=12)
plt.title('Video Diffusion Model 訓練損失', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

<a id="section7"></a>
## 7. 動画生成とGIF可視化

### 📊 逆拡散で動画を生成

In [ ]:
# ============================================================
# 動画サンプリング
# ============================================================

print("動画を生成中...")
generated = diffusion.sample((4, NUM_FRAMES, 1, 32, 32))
print(f"生成動画: {generated.shape}")

# フレーム列として可視化
def show_generated_videos(videos, title="生成された動画"):
    """生成動画のフレーム列を表示"""
    num_videos = videos.shape[0]
    T = videos.shape[1]
    frames_show = min(T, 8)
    
    fig, axes = plt.subplots(num_videos, frames_show, figsize=(2 * frames_show, 2 * num_videos))
    
    for i in range(num_videos):
        step = max(1, T // frames_show)
        for j in range(frames_show):
            t_idx = j * step
            frame = videos[i, t_idx, 0].cpu().numpy()
            axes[i, j].imshow(frame, cmap='gray', vmin=-1, vmax=1)
            axes[i, j].set_title(f't={t_idx}', fontsize=9)
            axes[i, j].axis('off')
    
    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

show_generated_videos(generated, "Video Diffusion Model で生成した動画")

In [ ]:
# ============================================================
# 拡散過程の可視化（ノイズ → 動画）
# ============================================================

def visualize_reverse_diffusion_process(diffusion_model, shape, steps_to_show=6):
    """逆拡散過程の途中ステップを可視化"""
    diffusion_model.model.eval()
    x = torch.randn(shape, device=diffusion_model.device)
    
    T_diff = diffusion_model.num_timesteps
    show_at = np.linspace(T_diff - 1, 0, steps_to_show, dtype=int)
    snapshots = {}
    
    for t in reversed(range(T_diff)):
        t_batch = torch.full((shape[0],), t, device=diffusion_model.device, dtype=torch.long)
        predicted_noise = diffusion_model.model(x, t_batch)
        
        alpha = diffusion_model.alphas[t]
        alpha_bar = diffusion_model.alphas_cumprod[t]
        beta = diffusion_model.betas[t]
        
        x = (1 / torch.sqrt(alpha)) * (x - beta / torch.sqrt(1 - alpha_bar) * predicted_noise)
        if t > 0:
            x = x + torch.sqrt(beta) * torch.randn_like(x)
        
        if t in show_at:
            snapshots[t] = x.clone().cpu()
    
    diffusion_model.model.train()
    
    # 可視化
    fig, axes = plt.subplots(steps_to_show, 4, figsize=(10, 2.5 * steps_to_show))
    
    for row, t in enumerate(sorted(snapshots.keys(), reverse=True)):
        video = snapshots[t][0]  # 最初のサンプル
        for col in range(4):
            frame_idx = col * (shape[1] // 4)
            axes[row, col].imshow(video[frame_idx, 0].numpy(), cmap='gray', vmin=-1, vmax=1)
            if row == 0:
                axes[row, col].set_title(f'Frame {frame_idx}', fontsize=10)
            axes[row, col].axis('off')
        axes[row, 0].set_ylabel(f't={t}', fontsize=10, fontweight='bold')
    
    plt.suptitle('逆拡散過程の可視化\n（ノイズ → 動画）', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualize_reverse_diffusion_process(diffusion, (1, NUM_FRAMES, 1, 32, 32))

In [ ]:
# ============================================================
# GIF出力（imageioが利用可能な場合）
# ============================================================

try:
    import imageio
    
    def save_video_as_gif(video_tensor, filepath, fps=8):
        """動画テンソルをGIFとして保存
        
        Args:
            video_tensor: (T, 1, H, W) or (T, H, W)
            filepath: 保存先パス
            fps: フレームレート
        """
        if video_tensor.dim() == 4:
            video_tensor = video_tensor[:, 0]
        
        frames = []
        for t in range(video_tensor.shape[0]):
            frame = video_tensor[t].cpu().numpy()
            frame = ((frame + 1) / 2 * 255).clip(0, 255).astype(np.uint8)
            frames.append(frame)
        
        imageio.mimsave(filepath, frames, fps=fps, loop=0)
        print(f"✅ GIF保存: {filepath}")
    
    # 生成動画をGIFで保存
    for i in range(min(2, generated.shape[0])):
        save_video_as_gif(
            generated[i], 
            f'/tmp/generated_video_{i}.gif',
            fps=4
        )
    print("GIFファイルを確認してみてください！")
    
except ImportError:
    print("⚠️ imageioがインストールされていません。")
    print("pip install imageio でインストールできます。")
    print("GIF保存はスキップしますが、フレーム表示は上で確認できます。")

<a id="summary"></a>
## 8. まとめと自己評価クイズ

### 🎯 このノートブックで学んだこと

**画像拡散から動画拡散への拡張**
- ✓ 既存のU-Netにtemporal層を追加するアプローチ
- ✓ (2+1)D Convolution: 空間と時間の分離畳み込み
- ✓ 時間方向のSelf-Attentionによる長距離時間依存の捕捉

**実装**
- ✓ TemporalUNetBlock, VideoUNet のスクラッチ実装
- ✓ Moving-MNISTデータセットの合成
- ✓ 動画拡散モデルの訓練パイプライン
- ✓ GIF可視化

### ⚠️ よくあるエラー

#### エラー #1: OOM (Out of Memory)
動画は画像の T 倍のメモリを使用します。  
**解決**: フレーム数(T)、解像度(H,W)、バッチサイズ(B)を小さくする。

#### エラー #2: 時間的一貫性がない生成
各フレームが独立に生成されているように見える場合。  
**解決**: Temporal Attentionの学習率を上げる、訓練エポック数を増やす。

---

## 🎓 自己評価クイズ

### Q1: (2+1)D Convolution の「2+1」は何を意味しますか？

<details>
<summary>💡 答えを見る</summary>

**答え**: 「2」は空間2次元（H×W）の畳み込み、「1」は時間1次元（T）の畳み込みです。
3D畳み込みを空間と時間に分離することで、パラメータ効率と学習安定性を向上させます。

</details>

---

### Q2: 動画拡散モデルで画像拡散モデルの事前学習重みを活用できるのはなぜ？

<details>
<summary>💡 答えを見る</summary>

**答え**: 空間処理の2D Conv部分は画像U-Netと同一構造のため、画像で学習した重みをそのまま使えます。
新規追加するtemporal層のみを初期化し、時間的な一貫性だけを追加学習すればよいからです。

</details>

---

### Q3: 動画用ノイズスケジュールで注意すべき点は？

<details>
<summary>💡 答えを見る</summary>

**答え**: 動画は画像よりも高次元なため、ノイズの影響が大きくなります。
Cosineスケジュールを使うことで、低ノイズ領域に多くのステップを割き、細かい時間構造を学習できます。

</details>

---

**次のステップ**: Notebook 132 では、U-NetからTransformerベースの **Diffusion Transformer (DiT)** へ進化させ、
Soraの技術的基盤を学びます！